# Week 6 In-Class Exercise: Decision Trees

In the textbook (Chapter 6), we saw how a **Decision Tree** learns a flowchart of yes/no questions: at each node it picks the feature and threshold that best separates the classes (minimizing **Gini impurity** or **entropy**). Trees are easy to read, need no feature scaling, and can overfit badly if left unchecked.

In this exercise you'll apply those **same ideas** to two datasets that are new to this course, so the lessons show up clearly instead of being washed out by data that is too easy:

> **Breast Cancer Wisconsin** — 569 tumors, each described by 30 numeric measurements of a cell-nucleus image (radius, texture, concavity, ...). The label is **malignant** or **benign**. This is a real diagnostic problem with genuine difficulty, so different tree settings give genuinely different results.
>
> **Diabetes** — 442 patients, 10 baseline measurements (BMI, blood pressure, serum values, ...). The target is a **continuous** number: disease progression one year later. We'll use this for the *regression* tree.

Every technique from the textbook transfers directly — adapt the code, don't invent new methods.

1. Load and explore the data
2. Train and **visualize** a Decision Tree
3. Read **class probabilities** from a leaf
4. Control overfitting with **`max_depth`** (regularization)
5. Compare the **`gini`** and **`entropy`** split criteria
6. Read the tree's **feature importances**
7. See that trees have **high variance**
8. Fit a **Decision Tree Regressor** on the diabetes data

### Useful API References

| Task | Documentation |
|------|---------------|
| Load classification data | [load_breast_cancer](https://scikit-learn.org/stable/modules/generated/sklearn.datasets.load_breast_cancer.html) |
| Load regression data | [load_diabetes](https://scikit-learn.org/stable/modules/generated/sklearn.datasets.load_diabetes.html) |
| Train/test split | [train_test_split](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.train_test_split.html) |
| Tree classifier | [DecisionTreeClassifier](https://scikit-learn.org/stable/modules/generated/sklearn.tree.DecisionTreeClassifier.html) |
| Tree regressor | [DecisionTreeRegressor](https://scikit-learn.org/stable/modules/generated/sklearn.tree.DecisionTreeRegressor.html) |
| Draw a tree | [plot_tree](https://scikit-learn.org/stable/modules/generated/sklearn.tree.plot_tree.html) |
| Cross-validation | [cross_val_score](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.cross_val_score.html) |
| Metrics | [accuracy_score](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.accuracy_score.html), [mean_squared_error](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.mean_squared_error.html) |

<table align="left">
  <td>
    <a href="https://colab.research.google.com/github/pjmcswee/IST707-Notebooks/blob/main/week6/week6_inclass_exercise.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>
  </td>
</table>

## Setup

Run this cell to import the libraries and set the plotting defaults.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

plt.rc('font', size=13)
plt.rc('axes', labelsize=13, titlesize=13)
plt.rc('legend', fontsize=11)
plt.rc('xtick', labelsize=10)
plt.rc('ytick', labelsize=10)

## Step 1: Load the Data (Provided)

The textbook Chapter 6 demo used the iris flowers. Here we load the **Breast Cancer Wisconsin** dataset that ships with scikit-learn. This cell is provided — just run it.

In [ ]:
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split

cancer = load_breast_cancer(as_frame=True)
X = cancer.data          # 30 numeric features
y = cancer.target        # 0 = malignant, 1 = benign
feature_names = list(cancer.feature_names)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y)

counts = np.bincount(y)
print('Total samples:', X.shape[0], ' Features:', X.shape[1])
print('Classes:', [str(name) for name in cancer.target_names])
print('Class balance (full set): '
      f'{cancer.target_names[0]}={int(counts[0])}, '
      f'{cancer.target_names[1]}={int(counts[1])}')
print('Train:', X_train.shape[0], ' Test:', X_test.shape[0])
X.head()

## Step 2: Train and Visualize a Decision Tree

A tree with no depth limit will grow until every leaf is pure. To *see* the logic, we start shallow. Train a `DecisionTreeClassifier(max_depth=3)` and draw it with `plot_tree`.

**Your turn!** Fit the tree on the training data, print train and test accuracy, and plot the tree.

**Hint (from the textbook):**
```python
from sklearn.tree import DecisionTreeClassifier, plot_tree
tree_clf = DecisionTreeClassifier(max_depth=3, random_state=42)
tree_clf.fit(X_train, y_train)
plot_tree(tree_clf, feature_names=feature_names,
          class_names=cancer.target_names, filled=True, rounded=True)
```

In [ ]:
# TODO: Fit a DecisionTreeClassifier(max_depth=3) on the training data.
#       Print the train accuracy and the test accuracy.
#       Then draw the tree with plot_tree.
# Your code here:

**Question:** Look at the very top node of the tree. Which single feature and threshold does the tree split on first? Why does it make sense that the tree considers this the most useful question to ask before any other?

*Your answer:*

## Step 3: Class Probabilities

A decision tree doesn't just output a label — each leaf holds the *proportion* of training samples of each class that landed there. `predict_proba` returns those proportions.

**Your turn!** Take the first 5 test samples and print, for each, the predicted class probabilities next to the predicted label and the true label.

**Hint (from the textbook):**
```python
tree_clf.predict_proba(X_test[:5]).round(3)
tree_clf.predict(X_test[:5])
```

In [ ]:
# TODO: For the first 5 test samples, print the predicted probabilities,
#       the predicted class, and the true class.
# Your code here:

**Question:** Some samples come back with a confident probability like 0.98, others with something closer to 0.7. Since every sample in the same leaf gets the *same* probability, what does a leaf with a 0.7 probability tell you about the training samples that fell into it?

*Your answer:*

## Step 4: Regularization with `max_depth`

An unconstrained tree keeps splitting until the leaves are pure — memorizing the training set. `max_depth` is the main dial for controlling this. Too shallow **underfits**; too deep **overfits**.

**Your turn!** For `max_depth` in `[1, 2, 3, 4, 5, 8, 12, None]`, fit a tree and record both **train** and **test** accuracy. Then plot both curves against depth.

**Hint:**
```python
depths = [1, 2, 3, 4, 5, 8, 12, None]
for d in depths:
    clf = DecisionTreeClassifier(max_depth=d, random_state=42).fit(X_train, y_train)
    # record accuracy_score on train and on test
```

In [ ]:
# TODO: Sweep max_depth over [1, 2, 3, 4, 5, 8, 12, None].
#       Collect train accuracy and test accuracy for each.
#       Plot both curves vs depth and print the table.
# Your code here:

**Question:** As `max_depth` grows, the train accuracy climbs all the way to 1.0 but the test accuracy peaks at a shallow depth and then *stops improving or drops*. Which depth would you pick, and what is the growing gap between the two curves telling you?

*Your answer:*

## Step 5: `gini` vs `entropy`

At each node the tree chooses the split that most reduces *impurity*. scikit-learn can measure impurity with **Gini** (the default) or **entropy** (information gain). They usually agree, but not always.

**Your turn!** Use `cross_val_score` (5-fold) on the training data to compare a tree built with `criterion='gini'` against one with `criterion='entropy'`. Print the mean CV accuracy of each.

**Hint:**
```python
from sklearn.model_selection import cross_val_score
for crit in ['gini', 'entropy']:
    scores = cross_val_score(
        DecisionTreeClassifier(criterion=crit, random_state=42),
        X_train, y_train, cv=5)
```

In [ ]:
# TODO: Compare criterion='gini' vs criterion='entropy' with 5-fold
#       cross_val_score on the training set. Print each mean CV accuracy.
# Your code here:

**Question:** Did the two criteria give the same CV accuracy or different ones? Gini is the default mainly because it is slightly cheaper to compute. Based on what you see here, is the *choice of criterion* a big lever or a small one compared to `max_depth` from Step 4?

*Your answer:*

## Step 6: Feature Importances

A trained tree can rank its features by how much each one reduced impurity across all its splits. This is one reason trees are prized for **interpretability**.

**Your turn!** Fit a `max_depth=4` tree, then plot a horizontal bar chart of the top 8 features by `feature_importances_`.

**Hint:**
```python
clf = DecisionTreeClassifier(max_depth=4, random_state=42).fit(X_train, y_train)
importances = clf.feature_importances_    # one value per feature, sums to 1
```

In [ ]:
# TODO: Fit a max_depth=4 tree. Sort features by feature_importances_
#       and plot the top 8 as a horizontal bar chart.
# Your code here:

**Question:** Most of the importance is concentrated in just a couple of features, while many of the 30 features get an importance of 0. What does an importance of 0 mean about how the tree used that feature? How could this ranking help a doctor or a data scientist who wants a simpler model?

*Your answer:*

## Step 7: Decision Trees Have High Variance

Trees are **unstable**: small changes in the training data can produce a very different tree. That instability is exactly why we later average many trees together in a Random Forest.

**Your turn!** Six times, draw a random 70% subsample of the training data, fit a `max_depth=3` tree, and record which feature it splits on *at the root*. Print the six root features.

**Hint:** the root split feature is `clf.tree_.feature[0]`, an index into `feature_names`.
```python
rng = np.random.RandomState(0)
idx = rng.choice(len(X_train), size=int(len(X_train) * 0.7), replace=False)
```

In [ ]:
# TODO: Six times, subsample 70% of the training rows, fit a
#       max_depth=3 tree, and print the feature it splits on at the root
#       (feature_names[clf.tree_.feature[0]]).
# Your code here:

**Question:** Did every subsample pick the *same* root feature, or did the root change from run to run? Connect what you see to the term **high variance**, and explain in one sentence why averaging many such trees (a Random Forest) would help.

*Your answer:*

## Step 8: A Regression Tree

Trees also do **regression**: instead of a class, each leaf predicts a *number* (the mean target of its training samples), which is why a regression tree's prediction looks like a **staircase**. We switch to the **diabetes** dataset and predict disease progression from a single feature, **BMI**, so we can see the staircase.

**Your turn!** Fit `DecisionTreeRegressor` at `max_depth` 2 and 5 on `bmi -> target`. Plot the data as a scatter and overlay each tree's prediction across the BMI range. Print the test RMSE for each depth.

**Hint (from the textbook):**
```python
from sklearn.tree import DecisionTreeRegressor
reg = DecisionTreeRegressor(max_depth=2, random_state=42).fit(X_bmi_train, y_train)
grid = np.linspace(bmi.min(), bmi.max(), 500).reshape(-1, 1)
y_line = reg.predict(grid)
```

In [ ]:
# TODO: Load diabetes, use the single 'bmi' feature to predict the target.
#       Fit DecisionTreeRegressor at max_depth=2 and max_depth=5.
#       Scatter the test data and overlay each tree's staircase prediction.
#       Print the test RMSE for each depth.
# Your code here:

**Question:** The `max_depth=2` line is a coarse staircase with a few wide steps; the `max_depth=5` line has many narrow steps that wiggle to chase individual points. Which one is starting to overfit, and how do the two test RMSE values back up your answer?

*Your answer:*

## Reflection Questions

Answer each question in the cell below it (1-3 sentences each).

**Q1:** A decision tree needs **no feature scaling**, while the SVMs and linear models from earlier weeks did. Looking at how a tree makes a split (a single `feature <= threshold` test), explain why scaling the features up or down doesn't change the tree it learns.

*Your answer:*

**Q2:** In Step 4 you saw a deep tree hit 100% train accuracy. Why is 100% training accuracy *not* something to celebrate, and which number were you actually watching to judge the model?

*Your answer:*

**Q3:** Trees split on one feature at a time, producing decision boundaries made of horizontal and vertical lines. What kind of true boundary (think of a diagonal separation between two classes) would force a tree to use many awkward steps to approximate it?

*Your answer:*

**Q4:** Step 7 showed that a single tree is high-variance. Briefly describe how a **Random Forest** (next week's topic) uses many trees to turn that weakness into a strength.

*Your answer:*